# Notebook 01 — Data Collection

IPL Cricket Performance Analysis — Cricsheet JSON pipeline.


Install dependencies once from the project root: `pip install -r requirements.txt`


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import zipfile
import json
from pathlib import Path
from collections import defaultdict
from scipy import stats

sns.set_theme(style="whitegrid")

BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
FIGURE_DIR = BASE_DIR / "outputs" / "figures"

for folder in [RAW_DIR, PROCESSED_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Libraries imported successfully!")
print("Project folders created successfully!")
print("Project location:", BASE_DIR)


Libraries imported successfully!
Project folders created successfully!
Project location: /Users/vishnu/my internship/Python dseh project


## Download Cricsheet IPL JSON ZIP


In [2]:
url = "https://cricsheet.org/downloads/ipl_json.zip"
zip_path = RAW_DIR / "ipl_json.zip"

if zip_path.exists():
    print("Using existing ZIP file:", zip_path)
else:
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    zip_path.write_bytes(response.content)
    print("IPL dataset downloaded successfully!")

zip_file = zipfile.ZipFile(zip_path)
print("File size:", round(zip_path.stat().st_size / (1024 * 1024), 2), "MB")


Using existing ZIP file: /Users/vishnu/my internship/Python dseh project/data/raw/ipl_json.zip
File size: 4.97 MB


In [3]:
bad_file = zip_file.testzip()

if bad_file:
    print("Problem found:", bad_file)
else:
    print("ZIP file verified successfully!")

json_files = [
    file for file in zip_file.namelist()
    if file.endswith(".json")
]

print("Total IPL match files:", len(json_files))
print("Sample filenames:")
for file in json_files[:5]:
    print(file)


ZIP file verified successfully!
Total IPL match files: 1243
Sample filenames:
1082591.json
1082592.json
1082593.json
1082594.json
1082595.json


## Season-wise match counts


In [4]:
season_counts = defaultdict(int)

for filename in json_files:
    with zip_file.open(filename) as file:
        match = json.load(file)

    info = match["info"]
    season = info.get("season", info["dates"][0][:4])
    season = int(str(season)[:4])
    season_counts[season] += 1

season_table = pd.DataFrame(
    list(season_counts.items()),
    columns=["Season", "Matches"]
).sort_values("Season")

display(season_table)

print("Total seasons:", len(season_table))
print("Total matches:", season_table["Matches"].sum())
print("Latest available season:", season_table["Season"].max())


,Season,Matches
10,2007,58
11,2009,117
12,2011,73
13,2012,74
14,2013,76
15,2014,60
16,2015,59
17,2016,60
0,2017,59
1,2018,60


Total seasons: 18
Total matches: 1243
Latest available season: 2026


## Sample JSON inspection


In [7]:
sample_filename = json_files[0]

with zip_file.open(sample_filename) as file:
    sample_match = json.load(file)

print("Match information:")
display(pd.Series(sample_match["info"]))

print("\nFirst innings information:")
display(sample_match["innings"][0])

print("\nFirst ball of the match:")
display(sample_match["innings"][0]["overs"][0]["deliveries"][0])


Match information:


balls_per_over                                                     6
city                                                       Hyderabad
dates                                                   [2017-04-05]
event              {'match_number': 1, 'name': 'Indian Premier Le...
gender                                                          male
match_type                                                       T20
officials          {'match_referees': ['J Srinath'], 'reserve_ump...
outcome            {'by': {'runs': 35}, 'winner': 'Sunrisers Hyde...
overs                                                             20
player_of_match                                       [Yuvraj Singh]
players            {'Royal Challengers Bangalore': ['CH Gayle', '...
registry           {'people': {'A Choudhary': '18e6906e', 'A Desh...
season                                                          2017
team_type                                                       club
teams              [Sunrisers Hyde


First innings information:


{'team': 'Sunrisers Hyderabad',
 'overs': [{'over': 0,
   'deliveries': [{'actual_delivery': '0.1',
     'batter': 'DA Warner',
     'bowler': 'TS Mills',
     'non_striker': 'S Dhawan',
     'runs': {'batter': 0, 'extras': 0, 'total': 0}},
    {'actual_delivery': '0.2',
     'batter': 'DA Warner',
     'bowler': 'TS Mills',
     'non_striker': 'S Dhawan',
     'runs': {'batter': 0, 'extras': 0, 'total': 0}},
    {'actual_delivery': '0.3',
     'batter': 'DA Warner',
     'bowler': 'TS Mills',
     'non_striker': 'S Dhawan',
     'runs': {'batter': 4, 'extras': 0, 'total': 4}},
    {'actual_delivery': '0.4',
     'batter': 'DA Warner',
     'bowler': 'TS Mills',
     'non_striker': 'S Dhawan',
     'runs': {'batter': 0, 'extras': 0, 'total': 0}},
    {'actual_delivery': '0.5',
     'batter': 'DA Warner',
     'bowler': 'TS Mills',
     'extras': {'wides': 2},
     'non_striker': 'S Dhawan',
     'runs': {'batter': 0, 'extras': 2, 'total': 2}},
    {'actual_delivery': '0.5',
     'batte


First ball of the match:


{'actual_delivery': '0.1',
 'batter': 'DA Warner',
 'bowler': 'TS Mills',
 'non_striker': 'S Dhawan',
 'runs': {'batter': 0, 'extras': 0, 'total': 0}}

## `extract_ipl_innings()` — original extraction logic


In [9]:
def extract_ipl_innings(archive):
    """Extract individual player batting innings from IPL JSON files."""

    innings_records = []
    appearances = defaultdict(
        lambda: {"matches": set(), "teams": set()}
    )

    for filename in archive.namelist():
        if not filename.endswith(".json"):
            continue

        with archive.open(filename) as file:
            match = json.load(file)

        info = match["info"]
        season = int(str(
            info.get("season", info["dates"][0][:4])
        )[:4])

        match_id = Path(filename).stem
        match_date = str(info["dates"][0])
        venue = info.get("venue", "Unknown")
        teams = info["teams"]

        for team, players in info.get("players", {}).items():
            for player in players:
                appearances[(season, player)]["matches"].add(match_id)
                appearances[(season, player)]["teams"].add(team)

        for innings_number, innings in enumerate(
            match["innings"], start=1
        ):
            batting_team = innings["team"]

            opponent = next(
                (team for team in teams if team != batting_team),
                "Unknown"
            )

            players_stats = {}
            batting_order = []

            def add_player(player):
                if player not in players_stats:
                    batting_order.append(player)
                    players_stats[player] = {
                        "Runs": 0,
                        "Balls_Faced": 0,
                        "Fours": 0,
                        "Sixes": 0,
                        "Dot_Balls": 0,
                        "Dismissals": 0,
                        "Powerplay_Runs": 0,
                        "Powerplay_Balls": 0,
                        "Middle_Overs_Runs": 0,
                        "Middle_Overs_Balls": 0,
                        "Death_Overs_Runs": 0,
                        "Death_Overs_Balls": 0
                    }

            for over_index, over in enumerate(innings["overs"]):
                over_number = over.get("over", over_index)

                if over_number <= 5:
                    phase = "Powerplay"
                elif over_number <= 14:
                    phase = "Middle_Overs"
                else:
                    phase = "Death_Overs"

                for delivery in over["deliveries"]:
                    batter = delivery["batter"]
                    non_striker = delivery["non_striker"]

                    add_player(batter)
                    add_player(non_striker)

                    batter_runs = delivery["runs"]["batter"]
                    extras = delivery.get("extras", {})
                    player = players_stats[batter]

                    player["Runs"] += batter_runs
                    player[f"{phase}_Runs"] += batter_runs

                    if "wides" not in extras:
                        player["Balls_Faced"] += 1
                        player[f"{phase}_Balls"] += 1

                        if batter_runs == 0:
                            player["Dot_Balls"] += 1

                    if not delivery["runs"].get("non_boundary", False):
                        if batter_runs == 4:
                            player["Fours"] += 1
                        elif batter_runs == 6:
                            player["Sixes"] += 1

                    for wicket in delivery.get("wickets", []):
                        dismissed = wicket["player_out"]
                        add_player(dismissed)

                        if wicket.get("kind", "").lower() != "retired hurt":
                            players_stats[dismissed]["Dismissals"] += 1

            for position, player_name in enumerate(
                batting_order, start=1
            ):
                player = players_stats[player_name]

                record = {
                    "Player": player_name,
                    "Season": season,
                    "Match_ID": match_id,
                    "Match_Date": match_date,
                    "Venue": venue,
                    "Batting_Team": batting_team,
                    "Opponent": opponent,
                    "Innings_Number": innings_number,
                    "Batting_Position": position,
                    "Runs": player["Runs"],
                    "Balls_Faced": player["Balls_Faced"],
                    "Fours": player["Fours"],
                    "Sixes": player["Sixes"],
                    "Dot_Balls": player["Dot_Balls"],
                    "Dismissals": player["Dismissals"],
                    "Not_Out": player["Dismissals"] == 0,
                    "Is_Super_Over": innings.get("super_over", False)
                }

                for phase in [
                    "Powerplay",
                    "Middle_Overs",
                    "Death_Overs"
                ]:
                    phase_runs = player[f"{phase}_Runs"]
                    phase_balls = player[f"{phase}_Balls"]

                    record[f"{phase}_Runs"] = phase_runs
                    record[f"{phase}_Balls"] = phase_balls

                    record[f"{phase}_Strike_Rate"] = (
                        phase_runs / phase_balls * 100
                        if phase_balls else np.nan
                    )

                balls = player["Balls_Faced"]

                record["Strike_Rate"] = (
                    player["Runs"] / balls * 100
                    if balls else np.nan
                )

                innings_records.append(record)

    return pd.DataFrame(innings_records), appearances


In [10]:
ipl_innings, player_appearances = extract_ipl_innings(zip_file)

print("Data extraction completed successfully!")
print("Total batting innings records:", len(ipl_innings))
print("Total player-season appearances:", len(player_appearances))

display(ipl_innings.head(10))


Data extraction completed successfully!
Total batting innings records: 19196
Total player-season appearances: 3238


,Player,Season,Match_ID,Match_Date,Venue,Batting_Team,Opponent,Innings_Number,Batting_Position,Runs,...,Powerplay_Runs,Powerplay_Balls,Powerplay_Strike_Rate,Middle_Overs_Runs,Middle_Overs_Balls,Middle_Overs_Strike_Rate,Death_Overs_Runs,Death_Overs_Balls,Death_Overs_Strike_Rate,Strike_Rate
0,DA Warner,2017,1082591,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,14,...,14,8,175.000000,0,0,NaN,0,0,NaN,175.000000
1,S Dhawan,2017,1082591,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,40,...,24,16,150.000000,16,15,106.666667,0,0,NaN,129.032258
2,MC Henriques,2017,1082591,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,52,...,17,13,130.769231,35,22,159.090909,0,2,0.000000,140.540541
3,Yuvraj Singh,2017,1082591,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,62,...,0,0,NaN,40,17,235.294118,22,10,220.000000,229.629630
4,DJ Hooda,2017,1082591,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,16,...,0,0,NaN,0,0,NaN,16,12,133.333333,133.333333
5,BCJ Cutting,2017,1082591,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,1,6,16,...,0,0,NaN,0,0,NaN,16,6,266.666667,266.666667
6,CH Gayle,2017,1082591,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,Sunrisers Hyderabad,2,1,32,...,26,18,144.444444,6,3,200.000000,0,0,NaN,152.380952
7,Mandeep Singh,2017,1082591,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,Sunrisers Hyderabad,2,2,24,...,24,16,150.000000,0,0,NaN,0,0,NaN,150.000000
8,TM Head,2017,1082591,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,Sunrisers Hyderabad,2,3,30,...,2,2,100.000000,28,20,140.000000,0,0,NaN,136.363636
9,KM Jadhav,2017,1082591,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,Sunrisers Hyderabad,2,4,31,...,0,0,NaN,31,16,193.750000,0,0,NaN,193.750000


## Player-season batting summary (original calculation)


In [11]:
regular_innings = ipl_innings[
    ~ipl_innings["Is_Super_Over"].astype(bool)
].copy()

batting_summary = regular_innings.groupby(
    ["Season", "Player"]
).agg(
    Innings=("Match_ID", "count"),
    Runs=("Runs", "sum"),
    Balls_Faced=("Balls_Faced", "sum"),
    Dismissals=("Dismissals", "sum"),
    Highest_Score=("Runs", "max"),
    Scores_30_Plus=("Runs", lambda x: (x >= 30).sum()),
    Scores_50_Plus=("Runs", lambda x: (x >= 50).sum()),
    Scores_100_Plus=("Runs", lambda x: (x >= 100).sum()),
    Fours=("Fours", "sum"),
    Sixes=("Sixes", "sum"),
    Dot_Balls=("Dot_Balls", "sum"),
    Powerplay_Runs=("Powerplay_Runs", "sum"),
    Powerplay_Balls=("Powerplay_Balls", "sum"),
    Middle_Overs_Runs=("Middle_Overs_Runs", "sum"),
    Middle_Overs_Balls=("Middle_Overs_Balls", "sum"),
    Death_Overs_Runs=("Death_Overs_Runs", "sum"),
    Death_Overs_Balls=("Death_Overs_Balls", "sum")
)

summary_lookup = batting_summary.to_dict("index")

records = []

for (season, player), appearance in player_appearances.items():
    batting = summary_lookup.get((season, player), {})

    runs = int(batting.get("Runs", 0))
    balls = int(batting.get("Balls_Faced", 0))
    dismissals = int(batting.get("Dismissals", 0))
    innings = int(batting.get("Innings", 0))
    fours = int(batting.get("Fours", 0))
    sixes = int(batting.get("Sixes", 0))
    dot_balls = int(batting.get("Dot_Balls", 0))

    matches = len(appearance["matches"])
    boundaries = fours + sixes
    boundary_runs = 4 * fours + 6 * sixes

    record = {
        "Player": player,
        "Season": season,
        "Teams_Played": ", ".join(sorted(appearance["teams"])),
        "Matches_Played": matches,
        "Innings": innings,
        "Runs": runs,
        "Balls_Faced": balls,
        "Dismissals": dismissals,
        "Batting_Average": runs / dismissals if dismissals else np.nan,
        "Strike_Rate": runs / balls * 100 if balls else np.nan,
        "Highest_Score": int(batting.get("Highest_Score", 0)),
        "Scores_30_Plus": int(batting.get("Scores_30_Plus", 0)),
        "Scores_50_Plus": int(batting.get("Scores_50_Plus", 0)),
        "Scores_100_Plus": int(batting.get("Scores_100_Plus", 0)),
        "Fours": fours,
        "Sixes": sixes,
        "Dot_Balls": dot_balls,
        "Boundary_Runs": boundary_runs,
        "Boundary_Percentage": boundaries / balls * 100 if balls else np.nan,
        "Dot_Ball_Percentage": dot_balls / balls * 100 if balls else np.nan,
        "Runs_Per_Match": runs / matches if matches else np.nan,
        "Runs_Per_Innings": runs / innings if innings else np.nan
    }

    for phase in ["Powerplay", "Middle_Overs", "Death_Overs"]:
        phase_runs = int(batting.get(f"{phase}_Runs", 0))
        phase_balls = int(batting.get(f"{phase}_Balls", 0))

        record[f"{phase}_Runs"] = phase_runs
        record[f"{phase}_Balls"] = phase_balls
        record[f"{phase}_Strike_Rate"] = (
            phase_runs / phase_balls * 100 if phase_balls else np.nan
        )

    records.append(record)

ipl_batting = pd.DataFrame(records)

ipl_batting = ipl_batting.sort_values(
    ["Season", "Runs"],
    ascending=[True, False]
).reset_index(drop=True)

print("Player-season statistics created!")
print("Total records:", len(ipl_batting))

display(ipl_batting.head(10))


Player-season statistics created!
Total records: 3238


,Player,Season,Teams_Played,Matches_Played,Innings,Runs,Balls_Faced,Dismissals,Batting_Average,Strike_Rate,...,Runs_Per_Innings,Powerplay_Runs,Powerplay_Balls,Powerplay_Strike_Rate,Middle_Overs_Runs,Middle_Overs_Balls,Middle_Overs_Strike_Rate,Death_Overs_Runs,Death_Overs_Balls,Death_Overs_Strike_Rate
0,SE Marsh,2007,Kings XI Punjab,11,11,616,441,9,68.444444,139.682540,...,56.000000,212,178,119.101124,353,226,156.194690,51,37,137.837838
1,G Gambhir,2007,Delhi Daredevils,14,14,534,379,13,41.076923,140.897098,...,38.142857,300,216,138.888889,210,147,142.857143,24,16,150.000000
2,ST Jayasuriya,2007,Mumbai Indians,14,14,514,309,12,42.833333,166.343042,...,36.714286,350,215,162.790698,164,94,174.468085,0,0,NaN
3,SR Watson,2007,Rajasthan Royals,15,15,472,311,10,47.200000,151.768489,...,31.466667,58,51,113.725490,351,226,155.309735,63,34,185.294118
4,GC Smith,2007,Rajasthan Royals,11,11,441,362,9,49.000000,121.823204,...,40.090909,181,176,102.840909,204,155,131.612903,56,31,180.645161
5,AC Gilchrist,2007,Deccan Chargers,14,14,436,318,13,33.538462,137.106918,...,31.142857,274,214,128.037383,162,104,155.769231,0,0,NaN
6,YK Pathan,2007,Rajasthan Royals,16,15,435,243,14,31.071429,179.012346,...,29.000000,115,73,157.534247,207,118,175.423729,113,52,217.307692
7,SK Raina,2007,Chennai Super Kings,16,14,421,295,11,38.272727,142.711864,...,30.071429,93,68,136.764706,281,193,145.595855,47,34,138.235294
8,MS Dhoni,2007,Chennai Super Kings,16,14,414,310,10,41.400000,133.548387,...,29.571429,0,1,0.000000,229,204,112.254902,185,105,176.190476
9,V Sehwag,2007,Delhi Daredevils,14,14,406,220,12,33.833333,184.545455,...,29.000000,236,142,166.197183,170,78,217.948718,0,0,NaN


## Save processed CSV files


In [12]:
summary_path = PROCESSED_DIR / "IPL_Batting_Statistics.csv"
innings_path = PROCESSED_DIR / "IPL_Batting_Innings.csv"

ipl_batting.to_csv(summary_path, index=False)
ipl_innings.to_csv(innings_path, index=False)

print("Both datasets saved successfully!")
print("Player-season dataset:", summary_path)
print("Individual innings dataset:", innings_path)


Both datasets saved successfully!
Player-season dataset: /Users/vishnu/my internship/Python dseh project/data/processed/IPL_Batting_Statistics.csv
Individual innings dataset: /Users/vishnu/my internship/Python dseh project/data/processed/IPL_Batting_Innings.csv
